In [2]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json

In [ ]:
class CariboDataFetcher:
    """Complete data acquisition and integration pipeline"""

    def __init__(self):
        self.weather_api = "https://archive-api.open-meteo.com/v1/archive"
        self.cariboo_locations = {
            'Williams Lake': (52.1411, -122.1430),  # Main city
            'Quesnel': (52.9707, -122.4926),        # Northern Cariboo
            'Lac La Hache': (51.5167, -121.2000),   # Southern Cariboo
        }
    def fetch_weather(self, location_name='Williams Lake', 
                        start_date='2012-01-01', end_date='2017-12-31'):
            """
            Fetch historical daily weather data from Open-Meteo API

            Parameters:
            - location_name: Name of location (Williams Lake, Quesnel, Lac La Hache)
            - start_date: Start date (YYYY-MM-DD)
            - end_date: End date (YYYY-MM-DD)

            Returns: pandas DataFrame with daily weather
            """

            if location_name not in self.cariboo_locations:
                raise ValueError(f"Location {location_name} not found")

            lat, lon = self.cariboo_locations[location_name]

            print(f"\nFetching weather for {location_name} ({lat}, {lon})...")
            print(f"Date range: {start_date} to {end_date}")

            params = {
                'latitude': lat,
                'longitude': lon,
                'start_date': start_date,
                'end_date': end_date,
                'daily': [
                    'temperature_2m_max',
                    'temperature_2m_mean',
                    'temperature_2m_min',
                    'relative_humidity_2m_max',
                    'relative_humidity_2m_mean',
                    'relative_humidity_2m_min',
                    'precipitation_sum',
                    'wind_speed_10m_max',
                    'pressure_msl_max',
                    'dew_point_2m_mean',
                ],
                'temperature_unit': 'celsius',
                'wind_speed_unit': 'kmh',
                'precipitation_unit': 'mm',
                'timezone': 'America/Vancouver'
            }

            try:
                response = requests.get(self.weather_api, params=params, timeout=30)
                response.raise_for_status()
                data = response.json()

                daily = data['daily']
                df = pd.DataFrame({
                    'Date': pd.to_datetime(daily['time']),
                    'Temp_Max_C': daily['temperature_2m_max'],
                    'Temp_Mean_C': daily['temperature_2m_mean'],
                    'Temp_Min_C': daily['temperature_2m_min'],
                    'Humidity_Max_Pct': daily['relative_humidity_2m_max'],
                    'Humidity_Mean_Pct': daily['relative_humidity_2m_mean'],
                    'Humidity_Min_Pct': daily['relative_humidity_2m_min'],
                    'Precip_mm': daily['precipitation_sum'],
                    'Wind_Speed_Max_kmh': daily['wind_speed_10m_max'],
                    'Pressure_Max_hPa': daily['pressure_msl_max'],
                    'Dew_Point_Mean_C': daily['dew_point_2m_mean'],
                    'Location': location_name
                })

                print(f"✓ Downloaded {len(df)} daily records")
                return df

            except Exception as e:
                print(f"✗ Error: {e}")
                return None
    def fetch_regional_weather(self, start_date='2012-01-01', end_date='2017-12-31'):
        """
        Fetch weather from all Cariboo locations and create regional average
        """

        all_dfs = []
        for location in self.cariboo_locations.keys():
            df = self.fetch_weather(location, start_date, end_date)
            if df is not None:
                all_dfs.append(df)

        # Combine and average across locations
        combined = pd.concat(all_dfs, ignore_index=True)
        regional = combined.groupby('Date').agg({
            'Temp_Max_C': 'mean',
            'Temp_Mean_C': 'mean',
            'Temp_Min_C': 'mean',
            'Humidity_Max_Pct': 'mean',
            'Humidity_Mean_Pct': 'mean',
            'Humidity_Min_Pct': 'mean',
            'Precip_mm': 'sum',
            'Wind_Speed_Max_kmh': 'mean',
            'Pressure_Max_hPa': 'mean',
            'Dew_Point_Mean_C': 'mean'
        }).reset_index()

        print(f"\n✓ Created regional weather dataset: {len(regional)} days from {len(self.cariboo_locations)} locations")
        return regional



In [4]:
CariboDataFetcher = CariboDataFetcher()
regional_weather_2012_2017 = CariboDataFetcher.fetch_regional_weather('2012-01-01', '2017-12-31')

AttributeError: 'CariboDataFetcher' object has no attribute 'fetch_regional_weather'